In [0]:
MY_ID = "2328120"
VOL = "/Volumes/workspace/default/logpulse"

print("Setup complete")
print(VOL)

In [0]:
spark.sql("CREATE VOLUME workspace.default.logpulse")
VOL = "/Volumes/workspace/default/logpulse"
print(VOL)

In [0]:
EP = ("'/api/checkout','/api/search','/api/cart','/api/login','/api/pay','/api/ship',"
      "'/api/track','/health','/static/app.js','/api/order/','/api/product/','/api/user/'")

b = spark.range(2160000).selectExpr(
    "id",
    "cast(pmod(id, 12) as int) e",
    "cast(pmod(id div 12, 720) as int) h",
    "cast(id div 8640 as int) j"
)

b = b.selectExpr(
    "*",
    f"element_at(array({EP}), e + 1) ep",
    "cast(if(id < 648000, pmod(id*7919, 648000), 999999) as int) dk",
    "timestamp'2025-03-01 00:00:00' + make_dt_interval(0, h) ts"
)

b = b.selectExpr(
    "*",
    "if(ep like '%/', concat(ep, cast(10000 + pmod(j*720 + h, 9000) as string)), ep) path",
    "if(e = 0 and h >= 321 and h <= 338, 30,"
    " if(e = 1 and h >= 472, 8, 1 + pmod(h + e, 3))) nerr"
)

b = b.selectExpr(
    "id",
    "dk",
    "path",
    "cast(if(j >= 250 - nerr, 503, 200) as string) st",
    "if(j < 75, format_string('10.9.0.%d', pmod(h*12 + e, 50) + 1),"
    " format_string('10.20.%d.%d', pmod(id, 4950) div 225,"
    " pmod(pmod(id, 4950), 225) + 1)) ip",
    "concat('[', if(dk >= 8600 and dk < 9800, '--/---/----:--:--:--',"
    " date_format(ts, 'dd/MMM/yyyy:HH:mm:ss')), ' +0000]') tsb"
)

LN = (
    "if(dk < 4000, concat_ws(' ', ip, '- -', tsb), concat_ws(' ', ip, '- -', tsb,"
    " case when dk >= 4000 and dk < 5800 then concat('GET ', path, ' HTTP/1.1')"
    " when dk >= 5800 and dk < 6100 then concat(char(34), 'GET ', path, ' HTTP/1.1')"
    " else concat(char(34), 'GET ', path, ' HTTP/1.1', char(34)) end, st,"
    " if(dk >= 6100 and dk < 8600, '-', cast(500 + pmod(id, 9500) as string)),"
    " format_string('R%09d', id))) value"
)

lg = b.selectExpr(LN, "dk")

(
    lg.select("value")
      .union(lg.filter("dk >= 9800 and dk < 15800").select("value"))
      .union(spark.range(900).selectExpr("'' value"))
      .write.mode("overwrite")
      .text(f"{VOL}/raw/access_logs")
)

print("Dataset generated!")

In [0]:
from pyspark.sql import functions as F

bronze = (
    spark.read.text(f"{VOL}/raw/access_logs")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_line_length", F.length("value"))
)

print("Bronze rows:", bronze.count())

In [0]:
from pyspark.sql import functions as F

pattern = r'^(\S+) - - \[([^\]]+)\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+)\s+(\S+)$'

silver_parsed = (
    bronze
    .withColumn("ip", F.regexp_extract("value", pattern, 1))
    .withColumn("timestamp_raw", F.regexp_extract("value", pattern, 2))
    .withColumn("method", F.regexp_extract("value", pattern, 3))
    .withColumn("path", F.regexp_extract("value", pattern, 4))
    .withColumn("protocol", F.regexp_extract("value", pattern, 5))
    .withColumn("status", F.regexp_extract("value", pattern, 6))
    .withColumn("bytes_raw", F.regexp_extract("value", pattern, 7))
    .withColumn("request_id", F.regexp_extract("value", pattern, 8))
)

parsed = silver_parsed.filter(F.col("status") != "")
unparsed = silver_parsed.filter(F.col("status") == "")

print("Parsed:", parsed.count())
print("Unparsed:", unparsed.count())

In [0]:
silver_dedup = parsed.dropDuplicates(["request_id"])

print("Before:", parsed.count())
print("After:", silver_dedup.count())

In [0]:
silver = (
    silver_dedup
    .withColumn(
        "event_ts",
        F.expr("try_to_timestamp(timestamp_raw, 'dd/MMM/yyyy:HH:mm:ss Z')")
    )
    .withColumn(
        "bytes",
        F.when(F.col("bytes_raw") == "-", None)
         .otherwise(F.col("bytes_raw").cast("long"))
    )
)

print("Bad timestamps:", silver.filter(F.col("event_ts").isNull()).count())
print("Missing bytes:", silver.filter(F.col("bytes").isNull()).count())

In [0]:
silver = silver.withColumn(
    "endpoint",
    F.regexp_replace("path", r"/\d+", "/{id}")
)

print("Distinct raw paths:", silver.select("path").distinct().count())
print("Distinct endpoints:", silver.select("endpoint").distinct().count())

In [0]:
silver = silver.filter(F.col("event_ts").isNotNull())

print("Silver rows:", silver.count())

In [0]:
gold_endpoint_hour = (
    silver
    .groupBy(
        "endpoint",
        F.date_trunc("hour", "event_ts").alias("hour_ts")
    )
    .agg(
        F.count("*").alias("requests"),
        F.sum(F.when(F.col("status").cast("int") >= 500, 1).otherwise(0)).alias("errors_5xx"),
        F.sum(F.when(
            (F.col("status").cast("int") >= 400) &
            (F.col("status").cast("int") < 500), 1
        ).otherwise(0)).alias("errors_4xx"),
        F.countDistinct("ip").alias("distinct_ips"),
        F.sum("bytes").alias("bytes_sent"),
        F.sum(F.when(F.col("bytes").isNull(), 1).otherwise(0)).alias("bytes_missing")
    )
    .withColumn(
        "error_rate_5xx",
        F.col("errors_5xx") / F.col("requests")
    )
)

print("Gold rows:", gold_endpoint_hour.count())

In [0]:
gold_client_ip = (
    silver
    .groupBy("ip")
    .agg(
        F.count("*").alias("requests"),
        F.countDistinct("endpoint").alias("endpoints_hit"),
        F.min("event_ts").alias("first_seen"),
        F.max("event_ts").alias("last_seen")
    )
)

total_requests = silver.count()

gold_client_ip = gold_client_ip.withColumn(
    "pct_of_total",
    F.col("requests") / F.lit(total_requests) * 100
)

# Top 1% = top 50 IPs
from pyspark.sql.window import Window

w = Window.orderBy(F.col("requests").desc())

gold_client_ip = (
    gold_client_ip
    .withColumn("rank", F.row_number().over(w))
    .withColumn("is_top_1pct", F.col("rank") <= 50)
    .drop("rank")
)

print("Client IP rows:", gold_client_ip.count())

In [0]:
gold_client_ip.filter("is_top_1pct").agg(
    F.sum("requests").alias("top_50_requests"),
    F.sum("pct_of_total").alias("top_50_percentage")
).show()

In [0]:
gold_endpoint_hour.filter(
    F.col("errors_5xx") >= 5
).orderBy("hour_ts").show(20, False)

In [0]:
print("Bronze:", bronze.count())
print("Parsed:", parsed.count())
print("After dedupe:", silver_dedup.count())
print("Silver:", silver.count())
print("Endpoints:", silver.select("endpoint").distinct().count())
print("Raw paths:", silver.select("path").distinct().count())
print("Gold endpoint-hour:", gold_endpoint_hour.count())
print("Gold client-IP:", gold_client_ip.count())

In [0]:
gold_endpoint_hour.write.mode("overwrite").format("delta").save(
    f"{VOL}/gold_endpoint_hour"
)

gold_client_ip.write.mode("overwrite").format("delta").save(
    f"{VOL}/gold_client_ip"
)

print("Gold tables saved")

In [0]:
#Q1 — 5xx error rate per endpoint per hour
gold_endpoint_hour.orderBy(
    F.col("error_rate_5xx").desc()
).select(
    "endpoint",
    "hour_ts",
    "requests",
    "errors_5xx",
    "error_rate_5xx"
).show(20, False)